In [1]:
import pandas as pd
import numpy as np
from scipy.spatial import KDTree
from joblib import Parallel, delayed

# Funções

In [ ]:
def delete_hdf_table(file_path, key):
    with pd.HDFStore(file_path, mode='a') as store:
        if f'/{key}' in store.keys():
            store.remove(f'/{key}')
            print(f"Table '{key}' removed from HDF5.")
        else:
            print(f"Table '{key}' not found in HDF5.")

def read_hdf(file_path, key):
    return pd.read_hdf(file_path, mode = 'r', key=key, encoding='utf-8')

def show_hdf_tables(file_path):
    with pd.HDFStore(file_path) as store:
        keys = store.keys()
        print(f"Current tables in {file_path}:")
        for key in keys:
            print(f"  - {key.lstrip('/')}")

def write_hdf_chunks(df, path, key, chunksize=10_000_000):
    with pd.HDFStore(path, mode='a', complevel=9, complib='blosc:lz4') as store: # Open in read+write mode to append
        for i in range(0, len(df), chunksize):
            chunk = df.iloc[i:i+chunksize]
            append_mode = i != 0 # Append after the first chunk to avoid overwriting data from previous chunks
            store.append(
                key,
                chunk,
                format='table',
                data_columns=True,
                min_itemsize={'gauge_code': 20},
                encoding='utf-8',
                append=append_mode
            )
            del chunk  # free memory
            print(f"Written rows {i} to {min(i+chunksize-1, len(df))}")

def read_hdf_chunks(file_path, key, chunksize=1_000_000):
    """
    Read HDF file in chunks and concatenate into a single DataFrame.
    Only works if the HDF key is stored in 'table' format.
    
    Parameters:
    file_path (str): Path to the HDF5 file
    key (str): Key/table name to read
    chunksize (int): Number of rows per chunk
    
    Returns:
    pd.DataFrame: Concatenated DataFrame
    """
    try:
        with pd.HDFStore(file_path, mode='r+') as store:  # Changed to 'r+' (read/write) and automatically closes the store
            if store.get_storer(key).is_table:
                dfs = []
                i = 1
                for chunk in store.select(key, chunksize=chunksize):
                    dfs.append(chunk)
                    print(f"Chunk {i} with {len(chunk)} rows read.")
                    i += 1                
                if dfs:  # Check if any chunks were read
                    return pd.concat(dfs, ignore_index=True)
                else:
                    print("No data found or chunksize too large.")
                    return pd.DataFrame()
            else:
                # If not table format, read all at once
                print("Not in table format, reading all data at once.")
                return store.select(key)  # Convert to DataFrame
    except FileNotFoundError:
        print(f"File {file_path} not found.")
        return pd.DataFrame()
    except KeyError:
        print(f"Key {key} not found in file {file_path}.")
        return pd.DataFrame()
    except Exception as e:
        print(f"Error reading HDF file: {e}")
        return pd.DataFrame()

def idw_interpolation(latitude, longitude, df_temp_without_gauge, kdtree, p=2):
    row = [latitude, longitude]
    distances, indices = kdtree.query(row, k=5)
    weights = 1 / (distances + 1e-6) ** p
    values = df_temp_without_gauge.iloc[indices]['rain_mm'].values
    return (np.sum(weights * values) / np.sum(weights))

def max_consecutive_dry_days_numpy(rain_array):
    """Retorna o maior número de dias consecutivos com rain == 0.0.
       rain_array pode conter floats e np.nan (np.nan != 0.0)."""
    arr = np.asarray(rain_array)
    dry = (arr == 0.0)          # bool array: True onde seco
    if not dry.any():
        return 0
    padded = np.concatenate(([False], dry, [False]))
    diff = np.diff(padded.astype(int))
    starts = np.where(diff == 1)[0]
    ends = np.where(diff == -1)[0]
    lengths = ends - starts
    return int(lengths.max()) if lengths.size > 0 else 0

def compute_metrics(df, parallel=True, n_jobs=-1, tol=None):
    """
    df: DataFrame com colunas ['gauge_code', 'datetime', 'rain_mm'].
    parallel: usa joblib para processar grupos em paralelo.
    tol: se fornecido (float >=0), considera `np.isclose(val, 0.0, atol=tol)` para definir dia seco.
         Se tol is None, usa igualdade exata (val == 0.0).
    Retorna DataFrame indexed por (gauge_code, year) com colunas:
      'annual_rainfall_mm', 'active_days', 'consecutive_dry_days'
    """
    df = df.copy()
    df['year'] = df['datetime'].dt.year

    # agregados vetorizados (rápidos)
    agg = df.groupby(['gauge_code', 'year']).agg(
        annual_rainfall_mm=('rain_mm', 'sum'),
        active_days=('rain_mm', lambda x: (x >= 0).sum())
    )

    # função que calcula consecutivos considerando tol se fornecido
    if tol is None:
        def _calc_consec(arr): 
            return max_consecutive_dry_days_numpy(arr)
    else:
        def _calc_consec(arr):
            a = np.asarray(arr)
            dry = np.isclose(a, 0.0, atol=tol, rtol=0)
            if not np.any(dry):
                return 0
            padded = np.concatenate(([False], dry, [False]))
            diff = np.diff(padded.astype(int))
            starts = np.where(diff == 1)[0]
            ends = np.where(diff == -1)[0]
            lengths = ends - starts
            return int(lengths.max()) if lengths.size > 0 else 0

    # calcula consecutive_dry_days (paraleliza por grupo se solicitado)
    if parallel:
        groups = [g for _, g in df.groupby(['gauge_code', 'year'])]
        results = Parallel(n_jobs=n_jobs)(
            delayed(lambda g: _calc_consec(g['rain_mm'].to_numpy()))(g) for g in groups
        )
        index = [(g['gauge_code'].iloc[0], g['year'].iloc[0]) for g in groups]
        cons_series = pd.Series(results, index=pd.MultiIndex.from_tuples(index, names=['gauge_code', 'year']),
                                name='consecutive_dry_days')
    else:
        cons_series = df.groupby(['gauge_code', 'year'])['rain_mm'] \
                        .apply(lambda x: _calc_consec(x.to_numpy()))

    # juntar agregados
    result = agg.join(cons_series)
    # garantir tipos inteiros para consecutivos/active_days se quiser
    result['consecutive_dry_days'] = result['consecutive_dry_days'].astype(int)
    result['active_days'] = result['active_days'].astype(int)
    return result

def mark_outlier_rain_optimized(df, threshold_rain_mm=200):
    # Garantir que a coluna 'datetime' está no formato datetime
    df['datetime'] = pd.to_datetime(df['datetime'])
    
    # 1. Filtrar apenas registros com chuva acima do threshold
    high_rain_df = df[df['rain_mm'] > threshold_rain_mm].copy()
    
    if high_rain_df.empty:
        print("Nenhum registro com chuva acima do threshold encontrado.")
        df['outlier_status_1'] = 0
        return df
    
    # 2. Criar DataFrames com datas de ontem e amanhã para merge
    yesterday_df = high_rain_df[['gauge_code', 'datetime']].copy()
    yesterday_df['datetime'] = yesterday_df['datetime'] - pd.Timedelta(days=1)
    yesterday_df = yesterday_df.merge(
        df[['gauge_code', 'datetime', 'rain_mm']], 
        on=['gauge_code', 'datetime'], 
        how='left'
    ).rename(columns={'rain_mm': 'rain_mm_yesterday'}).fillna(0)
    
    tomorrow_df = high_rain_df[['gauge_code', 'datetime']].copy()
    tomorrow_df['datetime'] = tomorrow_df['datetime'] + pd.Timedelta(days=1)
    tomorrow_df = tomorrow_df.merge(
        df[['gauge_code', 'datetime', 'rain_mm']], 
        on=['gauge_code', 'datetime'], 
        how='left'
    ).rename(columns={'rain_mm': 'rain_mm_tomorrow'}).fillna(0)
    
    # 3. Juntar os dados adjacentes de volta ao high_rain_df
    high_rain_df = high_rain_df.merge(
        yesterday_df[['gauge_code', 'datetime', 'rain_mm_yesterday']],
        on=['gauge_code', 'datetime'],
        how='left'
    )
    
    high_rain_df = high_rain_df.merge(
        tomorrow_df[['gauge_code', 'datetime', 'rain_mm_tomorrow']],
        on=['gauge_code', 'datetime'],
        how='left'
    )
    
    # Preencher NaN com 0 caso não existam dados para os dias adjacentes
    high_rain_df[['rain_mm_yesterday', 'rain_mm_tomorrow']] = high_rain_df[
        ['rain_mm_yesterday', 'rain_mm_tomorrow']
    ].fillna(0)
    
    # 4. Calcular soma da chuva nos dias adjacentes
    high_rain_df['adjacent_days_mm'] = high_rain_df['rain_mm_yesterday'] + high_rain_df['rain_mm_tomorrow']
    
    # 5. Aplicar regra para identificar outlier
    condition = (high_rain_df['adjacent_days_mm'] < 0.025 * high_rain_df['rain_mm'])
    high_rain_df['outlier_status_1'] = np.where(condition, 1, 0)
    
    # 6. Mesclar resultados de volta ao DataFrame original
    df['outlier_status_1'] = 0
    
    # Atualizar apenas os registros que foram processados
    outlier_mask = df.index.isin(high_rain_df[high_rain_df['outlier_status_1'] == 1].index)
    df.loc[outlier_mask, 'outlier_status_1'] = 1

    df_outlier_filter_1_export = df[df['outlier_status_1'] == 1]
    df_outlier_filter_1_export.to_hdf(
        cleaned_path,
        key='table_outlier_filter_1_export',
        mode='r+', 
        append=False, 
        complevel=9, 
        complib='blosc:lz4', 
        format='table', 
        data_columns=True, 
        min_itemsize={'gauge_code': 20}
    )
    
    print(f"Encontrados {outlier_mask.sum()} outliers potenciais")
    print(df.head())

    return df

# Parâmetros

In [3]:
cleaned_path = './1 - Organized data gauge/BRAZIL/DATASETS/BRAZIL_DAILY_1961_2024_CLEANED.h5'
upper_threshold = 450  # mm/day
lower_threshold = 0  # mm/day
rainfall_threshold = 200.0  # mm

# Leitura de Dados

In [ ]:
df_data = read_hdf_chunks(cleaned_path, key='table_data', chunksize=1_000_000)
print(f"DataFrame loaded with {len(df_data)} rows.")
df_data = df_data[(df_data['rain_mm']>= lower_threshold)
                  & (df_data['rain_mm']<= upper_threshold)
                  ].reset_index(drop=True)
df_data

Chunk 1 with 1000000 rows read.
Chunk 2 with 1000000 rows read.
Chunk 3 with 1000000 rows read.
Chunk 4 with 1000000 rows read.
Chunk 5 with 1000000 rows read.
Chunk 6 with 1000000 rows read.
Chunk 7 with 1000000 rows read.
Chunk 8 with 1000000 rows read.
Chunk 9 with 1000000 rows read.
Chunk 10 with 1000000 rows read.
Chunk 11 with 1000000 rows read.
Chunk 12 with 1000000 rows read.
Chunk 13 with 1000000 rows read.
Chunk 14 with 1000000 rows read.
Chunk 15 with 1000000 rows read.
Chunk 16 with 1000000 rows read.
Chunk 17 with 1000000 rows read.
Chunk 18 with 1000000 rows read.
Chunk 19 with 1000000 rows read.
Chunk 20 with 1000000 rows read.
Chunk 21 with 1000000 rows read.
Chunk 22 with 1000000 rows read.
Chunk 23 with 1000000 rows read.
Chunk 24 with 1000000 rows read.
Chunk 25 with 1000000 rows read.
Chunk 26 with 1000000 rows read.
Chunk 27 with 1000000 rows read.
Chunk 28 with 1000000 rows read.
Chunk 29 with 1000000 rows read.
Chunk 30 with 1000000 rows read.
Chunk 31 with 10000

In [9]:
df_info = read_hdf(cleaned_path, key='table_info')
df_info

,gauge_code,state,city,name_station,lat,long,responsible,source,state_abbreviation
0,00047000,PARÁ,SALINÓPOLIS,SALINÓPOLIS,-0.650000,-47.550000,INMET,HIDROWEB,PA
1,00047002,PARÁ,SALINÓPOLIS,SALINÓPOLIS,-0.623100,-47.353600,ANA,HIDROWEB,PA
2,00047003,PARÁ,CURUÇA,CURUÇA,-0.737500,-47.853600,ANA,HIDROWEB,PA
3,00047004,PARÁ,PRIMAVERA,PRIMAVERA,-0.929400,-47.099400,ANA,HIDROWEB,PA
4,00047005,PARÁ,MARAPANIM,MARUDA,-0.633600,-47.658300,ANA,HIDROWEB,PA
...,...,...,...,...,...,...,...,...,...
18977,S713,MATO GROSSO DO SUL,NOVA ANDRADINA,NOVA ANDRADINA | S713,-22.078611,-53.465833,INMET,INMET,MS
18978,S714,MATO GROSSO DO SUL,PEDRO GOMES,PEDRO GOMES | S714,-18.072778,-54.548889,INMET,INMET,MS
18979,S715,MATO GROSSO DO SUL,RIBAS DO RIO PARDO,RIBAS DO RIO PARDO | S715,-20.466694,-53.763028,INMET,INMET,MS
18980,S716,MATO GROSSO DO SUL,SANTA RITA DO PARDO,SANTA RITA DO PARDO | S716,-21.305889,-52.820375,INMET,INMET,MS


In [13]:
df_preclassif = compute_metrics(df_data, parallel=True, n_jobs=-1, tol=None).reset_index(drop=False)
df_preclassif

,gauge_code,year,annual_rainfall_mm,active_days,consecutive_dry_days
0,00047000,1961,2186.0,365,275
1,00047000,1962,273.8,365,153
2,00047000,1963,686.2,365,115
3,00047000,1964,597.5,366,145
4,00047002,1977,133.4,23,6
...,...,...,...,...,...
345863,S713,2021,76.2,365,150
345864,S714,2021,828.0,365,75
345865,S715,2021,1041.8,365,76
345866,S716,2021,928.8,365,68


In [14]:
df_preclassif['preclassif'] = df_preclassif.apply(
    lambda row: 'LQ' if (row['annual_rainfall_mm'] < 300
                         or row['annual_rainfall_mm'] > 6000
                        #  or row['active_days'] < 305 # parâmetro aferido na parcela P e Q1 do Quality Index (Q)
                         or row['consecutive_dry_days'] > 200) else "", axis=1)
# df_preclassif.to_hdf(cleaned_path, key='table_preclassif', mode='r+', append = False, complevel=9, complib='blosc:lz4', encoding='utf-8')
write_hdf_chunks(df_preclassif, cleaned_path, key='table_preclassif', chunksize=10_000_000)
df_preclassif = read_hdf(cleaned_path, key='table_preclassif')
df_preclassif

# 346029 outliers potenciais

Written rows 0 to 345868


,gauge_code,year,annual_rainfall_mm,active_days,consecutive_dry_days,preclassif
0,00047000,1961,2186.0,365,275,LQ
1,00047000,1962,273.8,365,153,LQ
2,00047000,1963,686.2,365,115,
3,00047000,1964,597.5,366,145,
4,00047002,1977,133.4,23,6,LQ
...,...,...,...,...,...,...
345863,S713,2021,76.2,365,150,LQ
345864,S714,2021,828.0,365,75,
345865,S715,2021,1041.8,365,76,
345866,S716,2021,928.8,365,68,


In [5]:
df_preclassif = read_hdf(cleaned_path, key='table_preclassif')

In [6]:
show_hdf_tables(cleaned_path)

# try:
#     with pd.HDFStore(cleaned_path, 'r+') as store:
#         print(f"Listing tables in: {cleaned_path}")
#         keys = store.keys()
#         if not keys:
#             print("  No tables found in this HDFStore.")
#         else:
#             for key in keys:
#                 # The key includes a leading '/', which we can remove for clarity.
#                 print(f"  - {key.lstrip('/')}")

# except FileNotFoundError:
#     print(f"Error: The file '{cleaned_path}' was not found.")
# except (OSError, ValueError):
#     print(f"Error: Could not read '{cleaned_path}'. It may not be a pandas HDFStore file. Try the h5py method instead.")

Current tables in ./1 - Organized data gauge/BRAZIL/DATASETS/BRAZIL_DAILY_1961_2024_CLEANED.h5:
  - table_data
  - table_data_outlier_filtered
  - table_info
  - table_outlier
  - table_outlier_filter_1
  - table_outlier_filter_1_export
  - table_outlier_filter_2_export
  - table_preclassif


In [7]:
preclassif_counts = df_preclassif['preclassif'].value_counts()
print(preclassif_counts)

preclassif
      299755
LQ     46113
Name: count, dtype: int64


In [8]:
print(f"Percentage of LQ preclassification: {preclassif_counts['LQ'] / preclassif_counts.sum() * 100:.2f}%")

Percentage of LQ preclassification: 13.33%


# Outlier treatment

In [10]:
max_lengths = {col: df_info[col].astype(str).map(len).max() for col in df_info.columns} 
print(max_lengths)

{'gauge_code': 10, 'state': 19, 'city': 39, 'name_station': 56, 'lat': 12, 'long': 12, 'responsible': 74, 'source': 10, 'state_abbreviation': 2}


In [11]:
max_lengths = {col: df_data[col].astype(str).map(len).max() for col in df_data.columns} 
print(max_lengths)

{'gauge_code': 10, 'datetime': 10, 'rain_mm': 19}


In [12]:
# df_preclassif = pd.read_hdf(cleaned_path, key = 'table_preclassif', encoding = 'utf-8')
df_data['year'] = df_data['datetime'].dt.year
df_outlier = pd.merge(df_data, df_preclassif[['gauge_code', 'year', 'preclassif']], on = ['gauge_code', 'year'], how = 'left')
df_outlier = df_outlier[df_outlier['preclassif'] != "LQ"]
df_outlier.pop('preclassif')
df_outlier.pop('year')
df_outlier

,gauge_code,datetime,rain_mm
0,110018901A,2021-01-01,6.30
1,110018901A,2021-01-02,1.79
2,110018901A,2021-01-03,0.00
3,110018901A,2021-01-04,16.15
4,110018901A,2021-01-05,1.58
...,...,...,...
123595834,88690050,2023-12-27,0.00
123595835,88690050,2023-12-28,0.00
123595836,88690050,2023-12-29,2.00
123595837,88690050,2023-12-30,0.00


In [ ]:
write_hdf_chunks(df_outlier, cleaned_path, 'table_outlier', chunksize=1_000_000)
print("Outlier-filtered data written to HDF5.") # Final message to the user that the data has been written successfully to the file
show_hdf_tables(cleaned_path)

Written rows 0 to 999999
Written rows 1000000 to 1999999
Written rows 2000000 to 2999999
Written rows 3000000 to 3999999
Written rows 4000000 to 4999999
Written rows 5000000 to 5999999
Written rows 6000000 to 6999999
Written rows 7000000 to 7999999
Written rows 8000000 to 8999999
Written rows 9000000 to 9999999
Written rows 10000000 to 10999999
Written rows 11000000 to 11999999
Written rows 12000000 to 12999999
Written rows 13000000 to 13999999
Written rows 14000000 to 14999999
Written rows 15000000 to 15999999
Written rows 16000000 to 16999999
Written rows 17000000 to 17999999
Written rows 18000000 to 18999999
Written rows 19000000 to 19999999
Written rows 20000000 to 20999999
Written rows 21000000 to 21999999
Written rows 22000000 to 22999999
Written rows 23000000 to 23999999
Written rows 24000000 to 24999999
Written rows 25000000 to 25999999
Written rows 26000000 to 26999999
Written rows 27000000 to 27999999
Written rows 28000000 to 28999999
Written rows 29000000 to 29999999
Written

In [4]:
df_outlier = read_hdf_chunks(cleaned_path, key='table_outlier', chunksize=1_000_000)

Chunk 1 with 1000000 rows read.
Chunk 2 with 1000000 rows read.
Chunk 3 with 1000000 rows read.
Chunk 4 with 1000000 rows read.
Chunk 5 with 1000000 rows read.
Chunk 6 with 1000000 rows read.
Chunk 7 with 1000000 rows read.
Chunk 8 with 1000000 rows read.
Chunk 9 with 1000000 rows read.
Chunk 10 with 1000000 rows read.
Chunk 11 with 1000000 rows read.
Chunk 12 with 1000000 rows read.
Chunk 13 with 1000000 rows read.
Chunk 14 with 1000000 rows read.
Chunk 15 with 1000000 rows read.
Chunk 16 with 1000000 rows read.
Chunk 17 with 1000000 rows read.
Chunk 18 with 1000000 rows read.
Chunk 19 with 1000000 rows read.
Chunk 20 with 1000000 rows read.
Chunk 21 with 1000000 rows read.
Chunk 22 with 1000000 rows read.
Chunk 23 with 1000000 rows read.
Chunk 24 with 1000000 rows read.
Chunk 25 with 1000000 rows read.
Chunk 26 with 1000000 rows read.
Chunk 27 with 1000000 rows read.
Chunk 28 with 1000000 rows read.
Chunk 29 with 1000000 rows read.
Chunk 30 with 1000000 rows read.
Chunk 31 with 10000

In [5]:
df_outlier

,gauge_code,datetime,rain_mm
0,110018901A,2021-01-01,6.30
1,110018901A,2021-01-02,1.79
2,110018901A,2021-01-03,0.00
3,110018901A,2021-01-04,16.15
4,110018901A,2021-01-05,1.58
...,...,...,...
107975572,88690050,2023-12-27,0.00
107975573,88690050,2023-12-28,0.00
107975574,88690050,2023-12-29,2.00
107975575,88690050,2023-12-30,0.00


In [6]:
df_outlier_filter_1 = mark_outlier_rain_optimized(df_outlier) # Usar a função otimizada para marcar outliers do tipo 1

Encontrados 4226 outliers potenciais
   gauge_code   datetime  rain_mm  outlier_status_1
0  110018901A 2021-01-01     6.30                 1
1  110018901A 2021-01-02     1.79                 1
2  110018901A 2021-01-03     0.00                 1
3  110018901A 2021-01-04    16.15                 1
4  110018901A 2021-01-05     1.58                 1


In [8]:
df_outlier_filter_1 = df_outlier[df_outlier['outlier_status_1'] != 1].reset_index(drop=True) # Filtrar outliers do tipo 1
write_hdf_chunks(df_outlier_filter_1, cleaned_path, 'table_outlier_filter_1', chunksize=10_000_000) # Escrever em chunks para eficiência 
print("Outlier-filtered data written to HDF5.") # Mensagem final para o usuário que os dados foram escritos com sucesso no arquivo
print(show_hdf_tables(cleaned_path))
df_outlier_filter_1

Written rows 0 to 9999999
Written rows 10000000 to 19999999
Written rows 20000000 to 29999999
Written rows 30000000 to 39999999
Written rows 40000000 to 49999999
Written rows 50000000 to 59999999
Written rows 60000000 to 69999999
Written rows 70000000 to 79999999
Written rows 80000000 to 89999999
Written rows 90000000 to 99999999
Written rows 100000000 to 107975552
Outlier-filtered data written to HDF5.
Current tables in ./1 - Organized data gauge/BRAZIL/DATASETS/BRAZIL_DAILY_1961_2024_CLEANED.h5:
  - table_data
  - table_data_outlier_filtered
  - table_info
  - table_outlier
  - table_outlier_filter_1
  - table_outlier_filter_1_export
  - table_outlier_filter_2_export
  - table_preclassif
None


,gauge_code,datetime,rain_mm,outlier_status_1
0,110018901A,2021-01-01,6.30,0
1,110018901A,2021-01-02,1.79,0
2,110018901A,2021-01-03,0.00,0
3,110018901A,2021-01-04,16.15,0
4,110018901A,2021-01-05,1.58,0
...,...,...,...,...
107975547,88690050,2023-12-27,0.00,0
107975548,88690050,2023-12-28,0.00,0
107975549,88690050,2023-12-29,2.00,0
107975550,88690050,2023-12-30,0.00,0


In [12]:
df_info = read_hdf(cleaned_path, key='table_info')
df_outlier_filter_1 = pd.merge(df_outlier_filter_1[['gauge_code', 'datetime', 'rain_mm']], df_info[['gauge_code', 'lat', 'long']], on ='gauge_code', how='left')
df_outlier_filter_1 = df_outlier_filter_1.dropna().reset_index(drop=True, inplace=False)
df_outlier_filter_1

,gauge_code,datetime,rain_mm,lat,long
0,110018901A,2021-01-01,6.30,-11.683234,-61.182871
1,110018901A,2021-01-02,1.79,-11.683234,-61.182871
2,110018901A,2021-01-03,0.00,-11.683234,-61.182871
3,110018901A,2021-01-04,16.15,-11.683234,-61.182871
4,110018901A,2021-01-05,1.58,-11.683234,-61.182871
...,...,...,...,...,...
107975547,88690050,2023-12-27,0.00,-31.811100,-52.389200
107975548,88690050,2023-12-28,0.00,-31.811100,-52.389200
107975549,88690050,2023-12-29,2.00,-31.811100,-52.389200
107975550,88690050,2023-12-30,0.00,-31.811100,-52.389200


In [13]:
start_date = '1961-01-01'
end_date = '2024-12-31'

df_date_filter = df_outlier_filter_1.loc[(df_outlier_filter_1['datetime'] >= start_date) & (df_outlier_filter_1['datetime'] <= end_date)].sort_values('datetime', ignore_index=True, ascending=True)

df_date_filter = df_date_filter[df_date_filter['rain_mm'] > rainfall_threshold].reset_index(drop=True) # Filter for high rainfall values

# Get sorted unique dates
analysis_dates = df_date_filter['datetime'].unique().tolist() # Get unique dates for the analysis
del df_date_filter
analysis_dates.sort()

# Process each date
for current_date in analysis_dates[:]:
    # Filter data for current date
    daily_data = df_outlier_filter_1[df_outlier_filter_1['datetime'] == current_date]

    df_gauge_filter = daily_data[daily_data['rain_mm'] > rainfall_threshold].reset_index(drop=True) # Filter for high rainfall values

    gauge_codes = df_gauge_filter['gauge_code'].unique() # Get unique gauge codes for the current date
    
    date_results = []
    
    for gauge in gauge_codes:
        gauge_data = daily_data[daily_data['gauge_code'] == gauge].iloc[0]
        lat, lon = gauge_data['lat'], gauge_data['long']
        observed_rain = gauge_data['rain_mm']
        
        # Initialize result row
        result_row = {
            'gauge_code': gauge,
            'datetime': current_date,
            'lat': lat,
            'long': lon,
            'observed_rain_mm': observed_rain,
            'interpolated_rain_mm': np.nan
        }
        
        # Only interpolate for high rainfall values
        if observed_rain > rainfall_threshold:
            neighbor_data = daily_data[daily_data['gauge_code'] != gauge]
            
            if len(neighbor_data) > 0:
                kd_tree = KDTree(neighbor_data[['lat', 'long']].values)
                result_row['interpolated_rain_mm'] = idw_interpolation(lat, lon, neighbor_data, kd_tree)
        
        date_results.append(pd.DataFrame([result_row]))
    
    # Combine results for current date
    daily_results = pd.concat(date_results, ignore_index=True)
    
    # Save to HDF5 with proper configuration
    append_mode = False if current_date == analysis_dates[0] else True

    # storage_mode = 'r+'
    # append_mode = True
    
    daily_results.to_hdf(
        cleaned_path,
        key='table_outlier_filter_2_export',
        mode='r+',
        format='table',
        complevel=9,
        encoding='utf-8',
        append=append_mode,
        complib='blosc:lz4',
        min_itemsize={'gauge_code': 20}  # Adjust based on your max gauge code length
    )
    del daily_results  # free memory
    print(f"Saved results for {current_date.date()} to {cleaned_path}")

Saved results for 1961-01-02 to ./1 - Organized data gauge/BRAZIL/DATASETS/BRAZIL_DAILY_1961_2024_CLEANED.h5
Saved results for 1961-01-04 to ./1 - Organized data gauge/BRAZIL/DATASETS/BRAZIL_DAILY_1961_2024_CLEANED.h5
Saved results for 1961-01-05 to ./1 - Organized data gauge/BRAZIL/DATASETS/BRAZIL_DAILY_1961_2024_CLEANED.h5
Saved results for 1961-01-11 to ./1 - Organized data gauge/BRAZIL/DATASETS/BRAZIL_DAILY_1961_2024_CLEANED.h5
Saved results for 1961-01-12 to ./1 - Organized data gauge/BRAZIL/DATASETS/BRAZIL_DAILY_1961_2024_CLEANED.h5
Saved results for 1961-01-14 to ./1 - Organized data gauge/BRAZIL/DATASETS/BRAZIL_DAILY_1961_2024_CLEANED.h5
Saved results for 1961-01-23 to ./1 - Organized data gauge/BRAZIL/DATASETS/BRAZIL_DAILY_1961_2024_CLEANED.h5
Saved results for 1961-01-25 to ./1 - Organized data gauge/BRAZIL/DATASETS/BRAZIL_DAILY_1961_2024_CLEANED.h5
Saved results for 1961-01-26 to ./1 - Organized data gauge/BRAZIL/DATASETS/BRAZIL_DAILY_1961_2024_CLEANED.h5
Saved results for 1

In [14]:
df_outlier_2 = read_hdf(cleaned_path, key='table_outlier_filter_2_export')
df_outlier_2

,gauge_code,datetime,lat,long,observed_rain_mm,interpolated_rain_mm
0,02244065,1961-01-02,-22.170000,-44.636900,251.20,9.687361
0,02146011,1961-01-04,-21.833300,-46.900000,218.30,21.706906
0,02241011,1961-01-05,-22.066700,-41.600000,212.20,23.561928
0,02241011,1961-01-11,-22.066700,-41.600000,360.20,11.429093
0,02143012,1961-01-12,-21.750000,-43.333300,220.00,38.436259
...,...,...,...,...,...,...
0,330300501A,2024-12-25,-21.412338,-42.196813,438.20,1.211600
0,56991380,2024-12-26,-19.971900,-41.110600,255.80,0.039461
0,290650101A,2024-12-29,-12.669000,-38.543000,264.72,0.000000
1,56991380,2024-12-29,-19.971900,-41.110600,408.40,0.655309


In [15]:
df_outlier_filter_2_export = df_outlier_2[df_outlier_2['interpolated_rain_mm'] >= 0.0].reset_index(drop = True)
df_outlier_filter_2_export = df_outlier_filter_2_export[df_outlier_filter_2_export['interpolated_rain_mm'] >= 0.35 * df_outlier_filter_2_export['observed_rain_mm']]
df_outlier_filter_2_export['outlier_status_2'] = 1
df_outlier_filter_2_export

,gauge_code,datetime,lat,long,observed_rain_mm,interpolated_rain_mm,outlier_status_2
7,02346034,1961-01-23,-23.4167,-46.5667,200.30,128.216270,1
10,02346162,1961-01-26,-23.7000,-46.0667,231.00,182.781510,1
11,02346223,1961-01-26,-23.7000,-46.0167,272.60,151.228139,1
12,02345145,1961-01-27,-23.5997,-45.9089,201.40,71.139330,1
17,02447038,1961-02-16,-24.8000,-47.9667,252.60,113.109432,1
...,...,...,...,...,...,...,...
4432,65365801,2024-05-27,-26.1653,-51.2281,315.60,190.400175,1
4435,65803001,2024-05-27,-25.7883,-52.1147,353.20,126.110557,1
4496,62780700,2024-11-04,-21.8194,-48.8281,218.20,120.550931,1
4526,65824880,2024-12-09,-25.6406,-51.8603,209.60,78.153355,1


In [16]:
df_outlier_filter_2_export.to_hdf(
        cleaned_path,
        key='table_outlier_filter_2_export',
        mode='r+',
        format='table',
        complevel=9,
        encoding='utf-8',
        append=False,
        complib='blosc:lz4',
        min_itemsize={'gauge_code': 20}  # Adjust based on your max gauge code length
    )
df_outlier_filter_2_export

,gauge_code,datetime,lat,long,observed_rain_mm,interpolated_rain_mm,outlier_status_2
7,02346034,1961-01-23,-23.4167,-46.5667,200.30,128.216270,1
10,02346162,1961-01-26,-23.7000,-46.0667,231.00,182.781510,1
11,02346223,1961-01-26,-23.7000,-46.0167,272.60,151.228139,1
12,02345145,1961-01-27,-23.5997,-45.9089,201.40,71.139330,1
17,02447038,1961-02-16,-24.8000,-47.9667,252.60,113.109432,1
...,...,...,...,...,...,...,...
4432,65365801,2024-05-27,-26.1653,-51.2281,315.60,190.400175,1
4435,65803001,2024-05-27,-25.7883,-52.1147,353.20,126.110557,1
4496,62780700,2024-11-04,-21.8194,-48.8281,218.20,120.550931,1
4526,65824880,2024-12-09,-25.6406,-51.8603,209.60,78.153355,1


In [17]:
show_hdf_tables(cleaned_path)

Current tables in ./1 - Organized data gauge/BRAZIL/DATASETS/BRAZIL_DAILY_1961_2024_CLEANED.h5:
  - table_data
  - table_data_outlier_filtered
  - table_info
  - table_outlier
  - table_outlier_filter_1
  - table_outlier_filter_1_export
  - table_outlier_filter_2_export
  - table_preclassif


# Data Deletion

In [18]:
df_data = read_hdf_chunks(cleaned_path, key = 'table_data', chunksize=1_000_000)
df_data

Chunk 1 with 1000000 rows read.
Chunk 2 with 1000000 rows read.
Chunk 3 with 1000000 rows read.
Chunk 4 with 1000000 rows read.
Chunk 5 with 1000000 rows read.
Chunk 6 with 1000000 rows read.
Chunk 7 with 1000000 rows read.
Chunk 8 with 1000000 rows read.
Chunk 9 with 1000000 rows read.
Chunk 10 with 1000000 rows read.
Chunk 11 with 1000000 rows read.
Chunk 12 with 1000000 rows read.
Chunk 13 with 1000000 rows read.
Chunk 14 with 1000000 rows read.
Chunk 15 with 1000000 rows read.
Chunk 16 with 1000000 rows read.
Chunk 17 with 1000000 rows read.
Chunk 18 with 1000000 rows read.
Chunk 19 with 1000000 rows read.
Chunk 20 with 1000000 rows read.
Chunk 21 with 1000000 rows read.
Chunk 22 with 1000000 rows read.
Chunk 23 with 1000000 rows read.
Chunk 24 with 1000000 rows read.
Chunk 25 with 1000000 rows read.
Chunk 26 with 1000000 rows read.
Chunk 27 with 1000000 rows read.
Chunk 28 with 1000000 rows read.
Chunk 29 with 1000000 rows read.
Chunk 30 with 1000000 rows read.
Chunk 31 with 10000

,gauge_code,datetime,rain_mm
0,110018901A,2021-01-01,6.30
1,110018901A,2021-01-02,1.79
2,110018901A,2021-01-03,0.00
3,110018901A,2021-01-04,16.15
4,110018901A,2021-01-05,1.58
...,...,...,...
123595834,88690050,2023-12-27,0.00
123595835,88690050,2023-12-28,0.00
123595836,88690050,2023-12-29,2.00
123595837,88690050,2023-12-30,0.00


In [19]:
df_filter_1 = read_hdf(
    cleaned_path
    , key='table_outlier_filter_1_export'
)
df_filter_1 = df_filter_1[['gauge_code', 'datetime', 'outlier_status_1']]
df_filter_1

,gauge_code,datetime,outlier_status_1
0,110018901A,2021-01-01,1
1,110018901A,2021-01-02,1
2,110018901A,2021-01-03,1
3,110018901A,2021-01-04,1
4,110018901A,2021-01-05,1
...,...,...,...
4540,120010401A,2023-06-07,1
4541,120010401A,2023-06-08,1
4544,120010401A,2023-06-11,1
4545,120010401A,2023-06-12,1


In [20]:
df_filter_2 = read_hdf(cleaned_path, key='table_outlier_filter_2_export')
df_filter_2['outlier_status_2'] = 1
df_filter_2 = df_filter_2[['gauge_code', 'datetime', 'outlier_status_2']]
df_filter_2

,gauge_code,datetime,outlier_status_2
7,02346034,1961-01-23,1
10,02346162,1961-01-26,1
11,02346223,1961-01-26,1
12,02345145,1961-01-27,1
17,02447038,1961-02-16,1
...,...,...,...
4432,65365801,2024-05-27,1
4435,65803001,2024-05-27,1
4496,62780700,2024-11-04,1
4526,65824880,2024-12-09,1


In [21]:
df_data_filtered = pd.merge(df_data, df_filter_1, on = ['gauge_code', 'datetime'], how = 'left').merge(df_filter_2, on = ['gauge_code', 'datetime'], how = 'left')
count_status_1 = df_data_filtered['outlier_status_1'].value_counts()
count_status_2 = df_data_filtered['outlier_status_2'].value_counts()
print(count_status_1, count_status_2)

outlier_status_1
1.0    4226
Name: count, dtype: int64 outlier_status_2
1.0    1356
Name: count, dtype: int64


In [22]:
df_data_filtered

,gauge_code,datetime,rain_mm,outlier_status_1,outlier_status_2
0,110018901A,2021-01-01,6.30,1.0,NaN
1,110018901A,2021-01-02,1.79,1.0,NaN
2,110018901A,2021-01-03,0.00,1.0,NaN
3,110018901A,2021-01-04,16.15,1.0,NaN
4,110018901A,2021-01-05,1.58,1.0,NaN
...,...,...,...,...,...
123595834,88690050,2023-12-27,0.00,NaN,NaN
123595835,88690050,2023-12-28,0.00,NaN,NaN
123595836,88690050,2023-12-29,2.00,NaN,NaN
123595837,88690050,2023-12-30,0.00,NaN,NaN


In [23]:
df_data_filtered = df_data_filtered[(df_data_filtered['outlier_status_1'] != 1) & (df_data_filtered['outlier_status_2'] != 1)]
df_data_filtered = df_data_filtered[['gauge_code', 'datetime', 'rain_mm']].copy(deep = True)
# df_data_filtered.to_csv('./1 - Organized data gauge/BRAZIL/DATASETS/BRAZIL_DAILY_1961_2024_CLEANED.csv', index=False, encoding='utf-8')
df_data_filtered # Final message to the user that the data has been written successfully to the file    

,gauge_code,datetime,rain_mm
32,110018901A,2021-02-02,9.48
33,110018901A,2021-02-03,4.73
34,110018901A,2021-02-04,73.28
37,110018901A,2021-02-07,0.20
38,110018901A,2021-02-08,0.00
...,...,...,...
123595834,88690050,2023-12-27,0.00
123595835,88690050,2023-12-28,0.00
123595836,88690050,2023-12-29,2.00
123595837,88690050,2023-12-30,0.00


In [24]:
write_hdf_chunks(df_data_filtered, cleaned_path, 'table_data_outlier_filtered', chunksize=10_000_000)

Written rows 0 to 9999999
Written rows 10000000 to 19999999
Written rows 20000000 to 29999999
Written rows 30000000 to 39999999
Written rows 40000000 to 49999999
Written rows 50000000 to 59999999
Written rows 60000000 to 69999999
Written rows 70000000 to 79999999
Written rows 80000000 to 89999999
Written rows 90000000 to 99999999
Written rows 100000000 to 109999999
Written rows 110000000 to 119999999
Written rows 120000000 to 123590257


In [26]:
df_data_filtered['gauge_code'].nunique()

18370